# Lab 2 standalone Kaggle run

Import only this notebook. No Python files or repository checkout are needed. All executable source, including the Unicode decoder fix, is embedded and hash-checked. Course inputs and pretrained model files download automatically.

Before running:
1. Set a GPU accelerator in Session options (T4 x2 if available) and turn Internet on. The code uses one GPU, not combined VRAM.
2. In Add-ons > Secrets, create HF_TOKEN with your Hugging Face read token and enable it for this notebook. It must read both private adapters and the generated-image dataset. Never put the token in a cell or share it.
3. Keep this notebook and its outputs private because they contain restored private model artifacts and course work.
4. Use Save Version > Save & Run All for a fresh background run. It is a new execution, not a way to attach to an already-running Colab session.

The notebook runs Parts 1 and 3. Part 2 restores your original completed RTX 4090 adapters, 120 images and measured results; it does not repeat training or claim that the original training happened on Kaggle or on a 16-GB GPU. Human ratings, team review, permissions and final external submissions remain separate.

Results, traces and executed notebooks stay under /kaggle/working. Large base-model caches and the isolated Python environment stay under /kaggle/temp so they are not saved as notebook outputs. A cold run needs about 40 GB of available disk overall and at least 8 GiB of host RAM. Kaggle runtime and quota limits still apply. Check that BOTH REAL GPU PREFLIGHTS PASSED appears before leaving the run.

At the end, download the Lab2_All_Results ZIP from Kaggle's Output panel and the notebook with its outputs. No Colab download service or localhost link is used. Checkpoints resume only while the same working files and matching hardware/source identity remain available; a new Kaggle session does not automatically import old Colab results.

In [ ]:
from pathlib import Path, PurePosixPath
import base64, hashlib, io, json, os, stat, subprocess, sys, zipfile

SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"

"""Kaggle-specific secrets, temporary caches and persistent result delivery."""
import hashlib
import os
from pathlib import Path
import shutil
import zipfile


def configure_runtime(working=None, scratch=None, environ=None, secret_loader=None, smoke_only=False):
    """Configure before ML imports. Never prompt for a token during a batch run."""
    if smoke_only:
        return {"mode": "CPU smoke only; no Kaggle GPU execution is certified"}
    environ = os.environ if environ is None else environ
    working = Path(working or "/kaggle/working").resolve()
    scratch = Path(scratch or "/kaggle/temp/lab2-runtime").resolve()
    if not working.is_dir():
        raise RuntimeError("Import this notebook into Kaggle, select a GPU and enable Internet before running it.")
    if scratch == working or working in scratch.parents:
        raise ValueError("Large caches and environments must be outside Kaggle's saved output directory.")
    token = environ.get("HF_TOKEN", "").strip()
    if not token:
        try:
            if secret_loader is None:
                from kaggle_secrets import UserSecretsClient
                secret_loader = UserSecretsClient().get_secret
            token = secret_loader("HF_TOKEN").strip()
        except Exception:
            raise RuntimeError("In Kaggle, open Add-ons > Secrets, add HF_TOKEN with your Hugging Face read token, "
                               "enable it for this notebook, then rerun. Never paste the token into a cell.") from None
    if not token:
        raise RuntimeError("The HF_TOKEN secret is empty. Set a read token with access to your private Part 2 artifacts.")
    scratch.mkdir(parents=True, exist_ok=True)
    environ["HF_TOKEN"] = token
    # A T4 x2 runtime still uses one card. Two devices are not one 32-GB GPU.
    visible = environ.get("CUDA_VISIBLE_DEVICES", "0") or "0"
    environ["CUDA_VISIBLE_DEVICES"] = visible.split(",")[0]
    paths = {
        "HF_HOME": scratch / "huggingface",
        "HF_HUB_CACHE": scratch / "huggingface/hub",
        "HF_DATASETS_CACHE": scratch / "datasets",
        "TORCH_HOME": scratch / "torch",
        "XDG_CACHE_HOME": scratch / "cache",
        "UV_CACHE_DIR": scratch / "uv-cache",
        "UV_PYTHON_INSTALL_DIR": scratch / "uv-python",
        "LAB2_ENV_DIR": scratch / "environments",
        "LAB2_BOOTSTRAP_DIR": scratch / "bootstrap",
    }
    for name, path in paths.items():
        environ[name] = str(path)
    environ["LAB2_KAGGLE_WORKING"] = str(working)
    environ["LAB2_STANDALONE_PARENT"] = str(working)
    # Return paths and hardware selection, never the secret.
    return {"working": str(working), "temporary_cache": str(scratch),
            "visible_gpu": environ["CUDA_VISIBLE_DEVICES"], "secret_loaded": True}


def publish_zip(archive, working=None, display_link=True):
    """Keep the exact validated archive at the top of Kaggle's Output panel."""
    archive = Path(archive).resolve()
    working = Path(working or "/kaggle/working").resolve()
    if not working.is_dir() or not archive.is_file() or archive.suffix.lower() != ".zip":
        raise FileNotFoundError("A complete result ZIP and an existing output directory are required.")
    with zipfile.ZipFile(archive) as bundle:
        if bundle.testzip() is not None:
            raise ValueError("Result ZIP failed its integrity check; it was not published.")
    def checksum(path):
        value = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(2**20), b""):
                value.update(block)
        return value.hexdigest()
    destination = working / archive.name
    expected = checksum(archive)
    if destination.is_symlink():
        raise FileExistsError("The output destination is a symlink and was preserved.")
    if destination != archive:
        if destination.exists():
            if checksum(destination) != expected:
                raise FileExistsError("A different same-named result ZIP exists and was preserved.")
        else:
            if shutil.disk_usage(working).free < archive.stat().st_size + 64 * 2**20:
                raise RuntimeError("Not enough space for the Output-panel copy. The original ZIP is preserved.")
            shutil.copy2(archive, destination)
    if checksum(destination) != expected:
        raise ValueError("Output copy checksum mismatch. Keep the original ZIP for recovery.")
    print("KAGGLE OUTPUT ZIP:", destination.name)
    print("SHA256:", expected)
    print("Download this file from Kaggle's Output panel. Save the completed version with outputs.")
    if display_link:
        from IPython.display import FileLink, display
        # Kaggle's normal notebook cwd is /kaggle/working. The Output panel is
        # the reliable fallback if a frontend does not render a file link.
        if Path.cwd().resolve() == working:
            display(FileLink(destination.name))
    return destination

print("Kaggle setup:", configure_runtime(smoke_only=SMOKE_ONLY))

if not SMOKE_ONLY:
    try:
        visible_gpu = subprocess.check_output(["nvidia-smi", "-L"], text=True, timeout=15)
    except (OSError, subprocess.SubprocessError) as error:
        raise RuntimeError("In Kaggle Session options select a GPU accelerator, turn Internet on, then run again. A CPU runtime cannot run this experiment.") from error
    print(visible_gpu.strip())

# All helper source is embedded below. No repository or .py upload is needed.
BUNDLE_SHA256 = '20dc9139997bfc00c3a34c0cc40eee176c838dc8d922495a522ebb65f178ded2'
BUNDLE_BASE64 = (
    '...'
)

def unpack(data, expected_sha256, parent):
    """No pre-existing repo, git executable, or package import is required."""
    if hashlib.sha256(data).hexdigest() != expected_sha256:
        raise ValueError("The embedded notebook bundle is incomplete or changed.")
    root = Path(parent) / ("Lab2_Standalone_" + expected_sha256[:16])
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        if archive.testzip() is not None:
            raise ValueError("The embedded ZIP failed its integrity check.")
        manifest = json.loads(archive.read("STANDALONE_MANIFEST.json"))
        files = manifest["files"]
        if len(archive.namelist()) != len(set(archive.namelist())):
            raise ValueError("Duplicate archive paths are not permitted.")
        if set(archive.namelist()) != set(files) | {"STANDALONE_MANIFEST.json"}:
            raise ValueError("The archive does not match its file manifest.")
        # Check every path and hash before creating or changing any file.
        for member in archive.infolist():
            name = member.filename
            path = PurePosixPath(name)
            if path.is_absolute() or ".." in path.parts or "\\" in name or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("Unsafe embedded path.")
            if member.is_dir():
                raise ValueError("Only regular manifest files are permitted.")
            if name != "STANDALONE_MANIFEST.json" and hashlib.sha256(archive.read(name)).hexdigest() != files[name]:
                raise ValueError(f"Embedded source hash mismatch: {name}")
            target = root / name
            if target.is_symlink() or not target.resolve().is_relative_to(root.resolve()):
                raise ValueError("An extraction target escaped the standalone folder.")
            if target.is_file() and target.read_bytes() != archive.read(name):
                raise ValueError(f"Local edits at {target} were preserved. Use a new empty parent folder.")
        root.mkdir(parents=True, exist_ok=True)
        for name in archive.namelist():
            target = root / name
            if not target.is_file():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(name))
    return root

default_parent = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
parent = Path(os.environ.get("LAB2_STANDALONE_PARENT", str(default_parent))).expanduser().resolve()
ROOT = unpack(base64.b64decode(BUNDLE_BASE64), BUNDLE_SHA256, parent)
os.environ["LAB2_STANDALONE_ROOT"] = str(ROOT)
print("Embedded code verified:", BUNDLE_SHA256)
print("Source base commit:", json.loads((ROOT / "STANDALONE_MANIFEST.json").read_text())["source_commit"])

from pathlib import Path
import os, sys, subprocess, json, time

ROOT = Path(os.environ["LAB2_STANDALONE_ROOT"]).resolve()
SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
if not SMOKE_ONLY:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ["LAB2_DEVICE"] = "cpu" if SMOKE_ONLY else "cuda"

# This override exists only for a developer CPU smoke, never for a GPU run.
test_python = os.environ.get("LAB2_STANDALONE_TEST_PYTHON", "") if SMOKE_ONLY else ""
if test_python:
    PYTHON = Path(test_python).expanduser().absolute()
    if not PYTHON.is_file():
        raise FileNotFoundError("The explicit CPU test interpreter does not exist.")
    subprocess.run([str(PYTHON), str(ROOT / "tools/preflight.py")], cwd=ROOT, check=True)
    (ROOT / ".lab2_runtime_cpu.json").write_text(json.dumps({"python": str(PYTHON), "cuda_requested": False}))
else:
    subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                    "--device", "cpu" if SMOKE_ONLY else "cuda"], cwd=ROOT, check=True)
    marker = ROOT / (".lab2_runtime_cpu.json" if SMOKE_ONLY else ".lab2_runtime_cuda.json")
    PYTHON = Path(json.loads(marker.read_text())["python"])
SEED = 266
print("Mode:", "CPU SMOKE ONLY: model calls are skipped" if SMOKE_ONLY else "FULL MODEL RUN")
print("Working folder:", ROOT)
print("Pinned interpreter:", PYTHON)

def run_script(relative, *arguments):
    record = ROOT / "runtime_checks" / (Path(relative).stem + "_" + str(time.time_ns()) + ".memory.json")
    command = [str(PYTHON), "-u", str(ROOT / "tools/monitored_run.py"), "--record", str(record),
               "--", str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
        status = process.wait()
    if status:
        raise RuntimeError(f"{Path(relative).name} stopped with exit {status}. Saved log: {log}. Run the final ZIP cell to retain all partial results.")


## Download and verify the course inputs

The PDFs, question CSV and knowledge-base manifest use a pinned source revision and checked SHA-256 values. Executable source is already embedded; this cell does not download Python scripts. Completed unchanged input files are kept when a download needs retrying.

In [ ]:
run_script("tools/standalone_colab.py", "resources")
DATA = ROOT / "part1_rag/data"
os.environ["LAB2_DATA_DIR"] = str(DATA)
os.environ.setdefault("QUESTIONS_PATH", str(DATA / "Questions.csv"))
print("Inputs ready:", DATA)

## Prepare standard Python kernels

This cell starts a real standard Python kernel without reading the host's system, user or profile configuration. It checks execution and saves an inline plot before the longer regression tests. Wait for `STANDARD CHILD KERNEL STARTUP PASSED`. The plot is a diagnostic, not a lab result. This does not restart the main notebook, change its settings, remove saved results or repeat Part 2 training.

In [ ]:
run_script("tools/kernel_preflight.py")

## Run the offline regression checks

These temporary fixtures check structure, data handling, checkpoints and export. They are not pretrained-model quality scores. The actual model preflights follow later.

In [ ]:
for name in ["test_lab_completion.py", "test_structured_responses.py", "test_constrained_decoding.py",
             "test_overnight_resume.py", "test_part2_submission.py", "test_result_download.py", "test_rtx4090_workflow.py"]:
    run_script("tools/" + name)
run_script("tools/test_kaggle_standalone.py")

## Recover the original Part 2 without uploading any files

Use your own Hugging Face account's read token. It must be able to read both private adapters and the image dataset. Access is checked before expensive work. The original dataset split is restored as needed, then each saved adapter and generated image is checked against its recorded hash. Comparison panels are rebuilt from those same images, not generated again.

The measured values displayed in the saved Part 2 notebook are explicitly labeled as original RTX 4090 results. A completed restoration does not create new training evidence, human judgments, sharing rights or grader access.

In [ ]:
if SMOKE_ONLY:
    print("SKIPPED: private Hugging Face access, dataset download and original Part 2 restoration.")
    run_script("tools/standalone_colab.py", "record-part2")
else:
    temporary_token = False
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("Enable the HF_TOKEN Kaggle secret, then rerun the first setup cell. No interactive login is used.")
    try:
        run_script("tools/standalone_colab.py", "check-hub-access")
        run_script("part2_diffusion/src/prepare_data.py", "--output-dir", ROOT / "part2_diffusion/data/processed")
        run_script("tools/standalone_colab.py", "restore-part2")
        run_script("tools/standalone_colab.py", "record-part2")
        run_script("part3_agents/src/prepare_evaluation.py", "--download-adapter")
    finally:
        if temporary_token:
            os.environ.pop("HF_TOKEN", None)

## Check the real model paths on this GPU

The RAG model, image understanding, prompt rewriting, critic, image editor and domain adapter are exercised sequentially. Tiny diagnostic images are not final samples. Stay connected until this prints `BOTH REAL GPU PREFLIGHTS PASSED`. CPU smoke output is never accepted as a passed GPU preflight.

In [ ]:
if SMOKE_ONLY:
    print("SKIPPED: actual GPU model preflights. No Kaggle GPU readiness is certified.")
else:
    run_script("tools/overnight_preflight.py", "--part", "part1_rag", "--data-dir", DATA)
    run_script("tools/overnight_preflight.py", "--part", "part3_agents")
    print("BOTH REAL GPU PREFLIGHTS PASSED. The full sequential run starts next.")

## Run Part 1 and Part 3 and keep the executed notebooks

Part 1 produces the validated CSV and evaluates the dense baseline against the final system. Part 3 compares 15 scenarios with and without critique and ten safety cases for each. Each request keeps its own trace and image path. Models run sequentially; revisions are capped at two.

The embedded notebooks save after every cell under `output/executed`. Unchanged successful items resume after interruption while those files remain on the runtime. Errors stay visible and are not replaced by made-up successful results.

In [ ]:
run_script("tools/overnight_runner.py", *(["--smoke-only"] if SMOKE_ONLY else []))
print("Saved executed notebooks:", ROOT / "output/executed")

## Check the results and pending review

A successful script exit is not enough: saved model errors and disabled-model fixtures fail the computation check. The separate audit lists real ratings, reference review, team details and other submission requirements that still need attention. Reports remain drafts while those are pending.

In [ ]:
run_script("tools/standalone_colab.py", "complete", *(["--smoke-only"] if SMOKE_ONLY else []))
run_script("tools/prepare_rating_forms.py")
run_script("tools/final_submission_audit.py")
print("Computation record:", ROOT / "runtime_checks/standalone_completion.json")
print("The final cell below downloads your saved results; it does not submit them.")

## Save and download all available results

This export-only cell creates one ZIP with saved outputs for Parts 1, 2 and 3, executed notebooks, reports, adapters, images, source/configuration and logs. Missing evidence is listed in BUNDLE_MANIFEST.json, never invented. Download the timestamped Lab2_All_Results ZIP from Kaggle's Output panel. If a run fails, run this final cell on its own to keep partial evidence; no model work repeats. Save the completed notebook version with outputs, and do not make private artifacts public.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, importlib.util

if "ROOT" not in globals():
    parent = Path(os.environ.get("LAB2_STANDALONE_PARENT", "/kaggle/working"))
    explicit = os.environ.get("LAB2_STANDALONE_ROOT")
    candidates = [Path(explicit)] if explicit else sorted(parent.glob("Lab2_Standalone_*"))
    candidates = [p for p in candidates if (p / "tools/export_lab_results.py").is_file()]
    if len(candidates) != 1:
        raise RuntimeError("Set LAB2_STANDALONE_ROOT to the existing run folder. Saved results were not changed.")
    ROOT = candidates[0].resolve()
subprocess.run([sys.executable, str(ROOT / "tools/export_lab_results.py"), "--all"], cwd=ROOT, check=True)
bundle = json.loads((ROOT / "output/exports/latest_all_results.json").read_text())
spec = importlib.util.spec_from_file_location("lab2_kaggle_export", ROOT / "tools/kaggle_runtime.py")
delivery = importlib.util.module_from_spec(spec)
spec.loader.exec_module(delivery)
working = ROOT.parent if globals().get("SMOKE_ONLY", False) else Path("/kaggle/working")
delivery.publish_zip(bundle["archive"], working=working, display_link=not globals().get("SMOKE_ONLY", False))
print("Upload that ZIP in our chat after the run. This cell does not submit the lab.")